In [ ]:
# 14 — Gold: review sentiment and summaries per handyman
# ai_analyze_sentiment scores each distinct review text once (cached in review_sentiment), then each
# handyman gets sentiment shares over their latest 20 reviews and an LLM summary of strengths and recurring
# complaints (ai_query, Claude Haiku). Summaries are only regenerated for handymen with new reviews,
# so re-runs cost nothing when nothing changed.

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())          # pipeline_lib.py lives next to this notebook
import pipeline_lib as pl
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "manual")
RUN_ID = dbutils.widgets.get("run_id")          # tables are created by 10_export_raw

started = time.time()
pl.ensure_feedback_table(spark)
# Materialise the features once: refresh_summaries reads them twice (all handymen + those needing a summary)
pl.compute_feedback(spark, spark.table(pl.T_SILVER)).write.mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(pl.fq("_stage_feedback_features"))
features = spark.table(pl.fq("_stage_feedback_features"))
pl.merge_feedback(spark, pl.refresh_summaries(spark, features, throttle=False))

fb = spark.table(pl.T_GOLD_FEEDBACK)
eligible = F.col("recent_review_count") >= pl.SUMMARY_MIN_REVIEWS
n_rows, n_eligible, with_summary, bad_sentiment = fb.agg(
    F.count("*"), F.count_if(eligible), F.count_if(eligible & F.col("review_summary").isNotNull()),
    F.count_if((F.col("sentiment_score") < -1) | (F.col("sentiment_score") > 1))).first()
display(fb.select("handyman_user_id", "review_count", "sentiment_score", "review_summary").limit(10))

pl.log_step(spark, RUN_ID, "14_gold_feedback", started, rows_out=n_rows,
            rows_rejected=n_eligible - with_summary,
            checks={"summaries_for_95pct": n_eligible == 0 or with_summary / n_eligible >= 0.95,
                    "sentiment_in_range": bad_sentiment == 0})